# 🌀 Spirulae-Splat — Google Colab Runtime

Train a **3D Gaussian Splatting** model using [spirulae-splat](https://github.com/harry7557558/spirulae-splat).

**Requirements:** GPU runtime (T4 / L4 / A100 / H100). Go to **Runtime → Change runtime type → GPU**.

**What this notebook does:**
1. Installs pre-built wheels from GitHub Releases (no compilation — ~30 seconds)
2. Runs training on your dataset
3. Exports a `.ply` file to your Google Drive

---

## Step 0 — GPU Check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No GPU detected!\n"
        "Go to Runtime → Change runtime type → select T4 GPU (or better) → Save."
    )

gpu_name = torch.cuda.get_device_name(0)
print(f"✅ GPU: {gpu_name}")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA:    {torch.version.cuda}")

## Step 1 — Mount Google Drive

Your dataset and output `.ply` file will be read/written from Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Configure Paths

**Edit the variables below** to point to your dataset and choose where to save outputs.

In [ ]:
import os

# ── Edit these ────────────────────────────────────────────────────────────────

# Path to your COLMAP or Nerfstudio dataset folder on Google Drive
DATA_PATH = "/content/drive/MyDrive/datasets/my_scene"

# Where to save training outputs and the final .ply
OUTPUT_DIR = "/content/drive/MyDrive/spirulae_outputs"

# Training preset — one of:
#   3dgs | 3dgs-confined | 3dgs-open | 3dgs-confined-low-texture
#   3dgs-open-low-texture | 3dgs-centered-object | academic-baseline
PRESET = "3dgs"

# Number of training iterations (default 30000; reduce to 5000 for a quick test)
NUM_ITERATIONS = 30000

# ── Do not edit below this line ───────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.exists(DATA_PATH), (
    f"❌ Dataset not found at: {DATA_PATH}\n"
    "Please update DATA_PATH above to point to your dataset."
)

print(f"Dataset:    {DATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Preset:     {PRESET}")
print(f"Iterations: {NUM_ITERATIONS}")

## Step 3 — Install Pre-built Wheels

Installs pre-compiled wheels from GitHub Releases — **no compilation needed**, takes ~30 seconds.

> 💡 If you get an import error after installing, update `RELEASE_TAG` to the latest release number from the [Releases page](https://github.com/notlasandu/spirulae-splat/releases).

In [ ]:
# ── Update RELEASE_TAG after each new build ────────────────────────────────────
GITHUB_REPO = "notlasandu/spirulae-splat"
RELEASE_TAG = "build-1"   # ← update to the latest tag from the Releases page
# ──────────────────────────────────────────────────────────────────────────────

# Fetch the list of wheel files from the release via GitHub API
import urllib.request, json
api_url = f"https://api.github.com/repos/{GITHUB_REPO}/releases/tags/{RELEASE_TAG}"
with urllib.request.urlopen(api_url) as r:
    release_data = json.loads(r.read())

wheel_assets = [a for a in release_data["assets"] if a["name"].endswith(".whl")]
print("Found wheels in release:")
for a in wheel_assets:
    print(f"  {a['name']} ({a['size'] // 1024 // 1024} MB)")

# Download and install each wheel
for asset in wheel_assets:
    url = asset["browser_download_url"]
    name = asset["name"]
    print(f"\nInstalling {name}...")
    !pip install "{url}" -q

print("\n✅ Wheels installed")

In [ ]:
# Install pure-Python runtime dependencies (fast — no compilation)
!pip install -q \
    jaxtyping \
    tyro \
    opencv-python-headless \
    plyfile \
    open3d \
    matplotlib \
    Pillow \
    rawpy \
    pytorch-msssim \
    "torchmetrics[image]" \
    typing_extensions \
    tabulate \
    scipy \
    imageio

print("✅ Runtime dependencies installed")

## Step 4 — Verify Installation

In [ ]:
# Verify the CUDA extension loaded correctly
try:
    import spirulae_splat
    from spirulae_splat import csrc as _C
    print("✅ spirulae_splat loaded successfully")
    print(f"   CUDA extension: {_C}")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("   Try updating RELEASE_TAG in Step 3 to a newer build.")
    raise

# Check entry point is registered
!spirulae-train --help 2>&1 | head -20

## Step 5 — Clone Repo (for export scripts)

The wheel contains the compiled Python package but not the utility scripts. We do a shallow clone so we can run `scripts/export_ply_3dgs.py` at the end. This is fast (~a few seconds) since `--depth 1` only fetches the latest commit.

In [ ]:
import os

REPO_PATH = "/content/spirulae-splat"

if not os.path.exists(REPO_PATH):
    !git clone --depth 1 -b spirulae-colab \
        https://github.com/notlasandu/spirulae-splat.git \
        "{REPO_PATH}"
    print("✅ Repo cloned (scripts only — no compilation needed)")
else:
    print("✅ Repo already cloned")

## Step 6 — Train

Runs training. Checkpoints are saved to `OUTPUT_DIR` on your Drive.

> The web viewer (port 7007) is disabled here since it's not accessible from Colab. Progress is shown in the cell output.

In [ ]:
import os

# Helps reduce VRAM fragmentation on smaller GPUs (T4 = 16 GB)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!spirulae-train {PRESET} \
    --data "{DATA_PATH}" \
    --output_dir_prefix "{OUTPUT_DIR}" \
    --num_iterations {NUM_ITERATIONS} \
    --viewer_port 0

## Step 7 — Export PLY

Finds the most recent training output folder and exports a `.ply` file to your Drive.

In [ ]:
import os, glob

# Find the most recently modified output folder
output_folders = sorted(
    glob.glob(f"{OUTPUT_DIR}/*/"),
    key=os.path.getmtime,
    reverse=True
)

assert output_folders, (
    f"❌ No output folders found in {OUTPUT_DIR}. "
    "Did training complete without errors?"
)

WORK_DIR = output_folders[0].rstrip("/")
print(f"Using output folder: {WORK_DIR}")

# Check that a checkpoint exists
checkpoints = sorted(glob.glob(f"{WORK_DIR}/*.ckpt"))
assert checkpoints, f"❌ No .ckpt files found in {WORK_DIR}"
print(f"Found checkpoint: {os.path.basename(checkpoints[-1])}")

# Run the export script
!python "{REPO_PATH}/scripts/export_ply_3dgs.py" "{WORK_DIR}"

# Report result
PLY_OUTPUT = f"{WORK_DIR}/splat.ply"
if os.path.exists(PLY_OUTPUT):
    size_mb = os.path.getsize(PLY_OUTPUT) / 1024 / 1024
    print(f"\n✅ PLY exported successfully!")
    print(f"   File: {PLY_OUTPUT}")
    print(f"   Size: {size_mb:.1f} MB")
    print(f"\n   View it at: https://supersplat.playcanvas.com/")
else:
    print("❌ PLY file not found — check the export output above for errors.")

---
## ✅ Done!

Your `.ply` file is saved to Google Drive at the path shown above.

**To view your splat:**
- [SuperSplat](https://supersplat.playcanvas.com/) — drag & drop the `.ply` file
- [3D Gaussian Splatting Viewer](https://antimatter15.com/splat/)

**To train on a new dataset:** edit the paths in **Step 2** and re-run from Step 2 onward.

**If the install fails after a Colab update:** update `RELEASE_TAG` in Step 3 to a fresh build — go to [Releases](https://github.com/notlasandu/spirulae-splat/releases) for the latest tag.